# RELLIS-3D five-class terrain segmentation

Train the compact U-Net on the official sequence split. Run this notebook on a GPU runtime in Colab or Kaggle.

The notebook owns experiment orchestration; reusable data, model, and metric code stays in `src`.

In [ ]:
import os
import random
import subprocess
import sys
from pathlib import Path

import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_ROOT = Path(os.environ.get("OFFROAD_REPO", "/content/offroad-terrain-route-planner"))
if not (REPO_ROOT / "src").exists():
    subprocess.run(
        ["git", "clone", "https://github.com/jayceeasparagus/offroad-terrain-route-planner.git", str(REPO_ROOT)],
        check=True,
    )
sys.path.insert(0, str(REPO_ROOT / "src"))

DATA_ROOT = Path(os.environ.get("RELLIS_ROOT", "/content/RELLIS-3D"))
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if DEVICE.type == "cuda":
    print(torch.cuda.get_device_name(0))
print("Dataset root:", DATA_ROOT)
assert DATA_ROOT.exists(), f"Set RELLIS_ROOT to the mounted dataset: {DATA_ROOT}"
assert (REPO_ROOT / "src").exists(), f"Repository not found: {REPO_ROOT}"


In [ ]:
from torch.utils.data import DataLoader

from offroad_perception.data import RellisSegmentationDataset, load_split_frames, load_taxonomy
from offroad_perception.models import CompactUNet
from offroad_perception.training import evaluate, train_one_epoch

taxonomy = load_taxonomy(REPO_ROOT / "configs/taxonomy.yaml")
train_frames = load_split_frames(DATA_ROOT, "train.lst")
val_frames = load_split_frames(DATA_ROOT, "val.lst")
test_frames = load_split_frames(DATA_ROOT, "test.lst")
print("Frames:", len(train_frames), len(val_frames), len(test_frames))
print("Classes:", taxonomy.names)


In [ ]:
IMAGE_SIZE = (320, 512)
BATCH_SIZE = 4
NUM_WORKERS = 2

train_dataset = RellisSegmentationDataset(train_frames, taxonomy, IMAGE_SIZE)
val_dataset = RellisSegmentationDataset(val_frames, taxonomy, IMAGE_SIZE)
test_dataset = RellisSegmentationDataset(test_frames, taxonomy, IMAGE_SIZE)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda"
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda"
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=DEVICE.type == "cuda"
)
batch = next(iter(train_loader))
print("Image batch:", batch["image"].shape)
print("Mask batch:", batch["mask"].shape)


In [ ]:
model = CompactUNet(in_channels=3, num_classes=taxonomy.num_classes, base_channels=16).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", patience=2, factor=0.5
)
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))


In [ ]:
EPOCHS = 20
checkpoint_dir = REPO_ROOT / "outputs/checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best_miou = -1.0
history = []

for epoch in range(1, EPOCHS + 1):
    train_result = train_one_epoch(
        model, train_loader, optimizer, DEVICE, taxonomy.num_classes, taxonomy.ignore_index
    )
    val_result = evaluate(
        model, val_loader, DEVICE, taxonomy.num_classes, taxonomy.ignore_index
    )
    scheduler.step(val_result.miou)
    record = {
        "epoch": epoch, "train_loss": train_result.loss,
        "val_loss": val_result.loss, "val_miou": val_result.miou,
        "val_per_class_iou": list(val_result.per_class_iou),
        "lr": optimizer.param_groups[0]["lr"],
    }
    history.append(record)
    print(
        f"Epoch {epoch:02d}/{EPOCHS} | train {train_result.loss:.4f} | "
        f"val {val_result.loss:.4f} | val mIoU {val_result.miou:.4f}"
    )
    checkpoint = {
        "model_state_dict": model.state_dict(), "taxonomy": taxonomy.names,
        "epoch": epoch, "history": history,
    }
    torch.save(checkpoint, checkpoint_dir / "latest_compact_unet.pt")
    if val_result.miou > best_miou:
        best_miou = val_result.miou
        torch.save(checkpoint, checkpoint_dir / "best_compact_unet.pt")
print("Best validation mIoU:", best_miou)


In [ ]:
import json

with (checkpoint_dir / "training_history.json").open("w", encoding="utf-8") as handle:
    json.dump(history, handle, indent=2)
print("Saved checkpoints to", checkpoint_dir)


In [ ]:
best_checkpoint = torch.load(
    checkpoint_dir / "best_compact_unet.pt", map_location=DEVICE
)
model.load_state_dict(best_checkpoint["model_state_dict"])
test_result = evaluate(
    model, test_loader, DEVICE, taxonomy.num_classes, taxonomy.ignore_index
)
print("Test loss:", test_result.loss)
print("Test mIoU:", test_result.miou)
for name, score in zip(taxonomy.names, test_result.per_class_iou):
    print(f"{name}: {score:.4f}")


## Training notes

- The dataset remains outside Git.
- Save checkpoints to Drive or Kaggle output after training.
- Do not report metrics until the test cell has run with the best validation checkpoint.
- The first run is a baseline; route planning and RGB-LiDAR fusion are implemented after segmentation is validated.